<a href="https://colab.research.google.com/github/elangbijak4/multi-agent-AI/blob/main/Hyperparameter_Tuning5_Agent_perbaikan_dead_lock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ============================================================
# GRID WORLD INTELLIGENT AGENT
# Anti-Deadlock Mechanisms with Conceptual Explanation
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
from collections import deque

from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# ============================================================
# 1. ENVIRONMENT DEFINITION
# ============================================================
# The environment is a 2D grid world.
# - EMPTY  : free space
# - DIRT   : positive target (to be cleaned)
# - BLOCK  : obstacle (must be avoided)
# - AGENT  : agent position (for visualization only)

GRID_SIZE = 10
EMPTY, DIRT, BLOCK, AGENT = 0, 1, -1, 2

np.random.seed(2)
grid = np.zeros((GRID_SIZE, GRID_SIZE), dtype=int)

# Place dirt (goals)
for _ in range(12):
    x, y = np.random.randint(0, GRID_SIZE, 2)
    grid[x, y] = DIRT

# Place blocks (constraints)
for _ in range(10):
    x, y = np.random.randint(0, GRID_SIZE, 2)
    if grid[x, y] == EMPTY:
        grid[x, y] = BLOCK

# Initial agent position
agent_pos = [0, 0]

# ============================================================
# 2. ACTION SPACE
# ============================================================
# The agent can move in four directions.
# Actions are discrete and symbolic,
# but will be chosen by a learned model.

ACTIONS = {
    0: (-1, 0),  # UP
    1: (1, 0),   # DOWN
    2: (0, -1),  # LEFT
    3: (0, 1)    # RIGHT
}

# ============================================================
# 3. PERCEPTION FUNCTION
# ============================================================
# The agent does NOT see the whole world.
# It only perceives its local neighborhood (partial observability).
#
# This is crucial:
# deadlocks emerge naturally from partial perception.

def perceive(grid, pos):
    """
    Local perception:
    The agent observes the 4 neighboring cells.
    """
    x, y = pos
    obs = []

    for dx, dy in ACTIONS.values():
        nx, ny = x + dx, y + dy
        if nx < 0 or ny < 0 or nx >= GRID_SIZE or ny >= GRID_SIZE:
            obs.append(BLOCK)  # world boundary treated as obstacle
        else:
            obs.append(grid[nx, ny])

    return np.array(obs)

# ============================================================
# 4. TRAINING DATA (SUPERVISED "WORLD KNOWLEDGE")
# ============================================================
# We give the agent a PRIOR KNOWLEDGE of how to act locally.
# This is NOT reinforcement learning.
#
# Think of this as:
# "how an experienced human would react to local perception".

X_train, y_train = [], []

for _ in range(800):
    obs = np.random.choice([EMPTY, DIRT, BLOCK], size=4)

    if DIRT in obs:
        # Move toward visible dirt
        action = int(np.where(obs == DIRT)[0][0])
    else:
        # Otherwise, avoid obstacles
        safe = [i for i, v in enumerate(obs) if v != BLOCK]
        action = np.random.choice(safe) if safe else 0

    X_train.append(obs)
    y_train.append(action)

X_train = np.array(X_train)
y_train = np.array(y_train)

# ============================================================
# 5. AGENT BRAIN (MODEL AS "OTAK")
# ============================================================
# You can swap this brain live in class to show different behaviors.

brain = DecisionTreeClassifier(max_depth=5)
brain.fit(X_train, y_train)

# ============================================================
# 6. SHORT-TERM MEMORY (ANTI-DEADLOCK CORE)
# ============================================================
# This memory enables the agent to recognize
# "I have been here before too many times."

VISIT_MEMORY = deque(maxlen=20)

# ============================================================
# 7. STEP FUNCTION WITH 4 ANTI-DEADLOCK MECHANISMS
# ============================================================
def step(grid, pos, epsilon=0.15):
    """
    One agent step with deadlock prevention.
    """

    # --------------------------------------------------------
    # (1) PERCEPTION
    # --------------------------------------------------------
    obs = perceive(grid, pos)

    # --------------------------------------------------------
    # (2) STATE SIGNATURE
    # --------------------------------------------------------
    # State = position + perception
    # This is the agent's INTERNAL notion of "where am I".
    state_signature = tuple(pos + obs.tolist())

    VISIT_MEMORY.append(state_signature)

    # --------------------------------------------------------
    # (3) DEADLOCK DETECTION (SHORT-TERM MEMORY)
    # --------------------------------------------------------
    # If the same state appears repeatedly,
    # the agent infers it is stuck in a loop.
    deadlock = VISIT_MEMORY.count(state_signature) > 2

    # --------------------------------------------------------
    # (4) ACTION SELECTION STRATEGY
    # --------------------------------------------------------
    # There are TWO ways the agent breaks deadlock:
    #
    # A. Stochastic exploration (epsilon)
    # B. Forced exploration when deadlock is detected

    if deadlock or np.random.rand() < epsilon:
        # ----------------------------------------------------
        # EXPLORATION MODE
        # ----------------------------------------------------
        # The agent deliberately tries something different,
        # but still avoids obviously impossible actions.
        safe_actions = []
        for a, (dx, dy) in ACTIONS.items():
            nx, ny = pos[0] + dx, pos[1] + dy
            if 0 <= nx < GRID_SIZE and 0 <= ny < GRID_SIZE:
                if grid[nx, ny] != BLOCK:
                    safe_actions.append(a)

        action = np.random.choice(safe_actions) if safe_actions else 0

    else:
        # ----------------------------------------------------
        # EXPLOITATION MODE
        # ----------------------------------------------------
        # Normal decision using the trained brain
        action = int(brain.predict([obs])[0])

    # --------------------------------------------------------
    # (5) EXECUTE ACTION
    # --------------------------------------------------------
    dx, dy = ACTIONS[action]
    nx, ny = pos[0] + dx, pos[1] + dy

    if nx < 0 or ny < 0 or nx >= GRID_SIZE or ny >= GRID_SIZE:
        return pos

    if grid[nx, ny] == BLOCK:
        return pos

    if grid[nx, ny] == DIRT:
        grid[nx, ny] = EMPTY

    return [nx, ny]

# ============================================================
# 8. LIVE ANIMATION
# ============================================================
fig, ax = plt.subplots(figsize=(5,5))
im = ax.imshow(grid, cmap="viridis", vmin=-1, vmax=2)

def update(frame):
    global agent_pos
    agent_pos = step(grid, agent_pos)

    display = grid.copy()
    display[agent_pos[0], agent_pos[1]] = AGENT
    im.set_data(display)
    ax.set_title(f"Step {frame}")
    return [im]

ani = animation.FuncAnimation(
    fig, update, frames=100, interval=300
)

plt.close()
HTML(ani.to_jshtml())
